In [4]:
import sqlite3
import pandas as pd
import logging
from ingestion_db import ingest_db  # Importing logic from your other script

# Setting up logs so you can track if the summary generation succeeds or fails
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(
    filename="logs/get_vendor_summary.log",
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode="a"
)

def create_vendor_summary(conn):
    '''this function will merge the different tables to get the overall vendor summary and adding new columns in the resultant data'''
    
    vendor_sales_summary = pd.read_sql_query("""WITH FreightSummary AS (
        SELECT
            VendorNumber,
            SUM(Freight) AS FreightCost
        FROM vendor_invoice
        GROUP BY VendorNumber
    ),
    
    PurchaseSummary AS (
        SELECT
            p.VendorNumber,
            p.VendorName,
            p.Brand,
            p.Description,
            p.PurchasePrice,
            pp.Price AS ActualPrice,
            pp.Volume,
            SUM(p.Quantity) AS TotalPurchaseQuantity,
            SUM(p.Dollars) AS TotalPurchaseDollars
        FROM purchases p
        JOIN purchase_prices pp
            ON p.Brand = pp.Brand
        WHERE p.PurchasePrice > 0
        GROUP BY p.VendorNumber, p.VendorName, p.Brand, p.Description, p.PurchasePrice, pp.Price, pp.Volume
    ),
    
    SalesSummary AS (
        SELECT
            VendorNo,
            Brand,
            SUM(SalesQuantity) AS TotalSalesQuantity,
            SUM(SalesDollars) AS TotalSalesDollars,
            SUM(SalesPrice) AS TotalSalesPrice,
            SUM(ExciseTax) AS TotalExciseTax
        FROM sales
        GROUP BY VendorNo, Brand
    )
    
    SELECT
        ps.VendorNumber,
        ps.VendorName,
        ps.Brand,
        ps.Description,
        ps.PurchasePrice,
        ps.ActualPrice,
        ps.Volume,
        ps.TotalPurchaseQuantity,
        ps.TotalPurchaseDollars,
        ss.TotalSalesQuantity,
        ss.TotalSalesDollars,
        ss.TotalSalesPrice,
        ss.TotalExciseTax,
        fs.FreightCost
    FROM PurchaseSummary ps
    LEFT JOIN SalesSummary ss
        ON ps.VendorNumber = ss.VendorNo
        AND ps.Brand = ss.Brand
    LEFT JOIN FreightSummary fs
        ON ps.VendorNumber = fs.VendorNumber
    ORDER BY ps.TotalPurchaseDollars DESC""", conn)
    
    return vendor_sales_summary

def clean_data(df):
    '''this function will clean the data'''
    
    # 1. Changing datatype to float
    # Ensuring Volume is a number so we can perform math on it
    df['Volume'] = df['Volume'].astype('float')
    
    # 2. Filling missing values with 0
    # If a product had 0 sales, SQL might return a "NaN" (Not a Number). 
    # We replace those with 0 so our calculations don't break.
    df.fillna(0, inplace=True)
    
    # 3. Removing spaces from categorical columns
    # Sometimes names have hidden spaces like " Heineken " which makes 
    # searching difficult. .strip() removes those leading/trailing spaces.
    df['VendorName'] = df['VendorName'].str.strip()
    df['Description'] = df['Description'].str.strip()
    
    # 4. Creating new columns for better analysis (Feature Engineering)
    
    # Gross Profit = Money from Sales - Money spent on Purchases
    df['GrossProfit'] = df['TotalSalesDollars'] - df['TotalPurchaseDollars']
    
    # Profit Margin = (Gross Profit / Total Sales) * 100
    # This shows what percentage of your revenue is actually profit.
    df['ProfitMargin'] = (df['GrossProfit'] / df['TotalSalesDollars']) * 100
    
    # Stock Turnover = Units Sold / Units Purchased
    # Helps identify if you are overstocking or selling through inventory quickly.
    df['StockTurnover'] = df['TotalSalesQuantity'] / df['TotalPurchaseQuantity']
    
    # Sales to Purchase Ratio = Revenue / Purchase Cost
    # A quick way to see the return on every dollar spent on stock.
    df['SalesToPurchaseRatio'] = df['TotalSalesDollars'] / df['TotalPurchaseDollars']
    
    return df

if __name__ == '__main__':
    # creating database connection
    conn = sqlite3.connect('inventory.db')

    logging.info('Creating Vendor Summary Table.....')
    summary_df = create_vendor_summary(conn)
    logging.info(summary_df.head())

    logging.info('Cleaning Data.....')
    clean_df = clean_data(summary_df)
    logging.info(clean_df.head())

    logging.info('Ingesting data.....')
    # This is likely the function from your ingestion_db.py script
    ingest_db(clean_df, 'vendor_sales_summary', conn) 
    logging.info('Completed!')